In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
from processing import  load_UCI_dataset, plot_regression_results, print_metrics, plot_convtran_loss
from UCI.pre_processing import  create_dataloaders

In [3]:
BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints"
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 625, STEP_SIZE= 312)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [02:29<00:00, 64.29it/s]


Skipped recordings: 81


100%|██████████| 1200/1200 [00:19<00:00, 62.11it/s]


Skipped recordings: 7


100%|██████████| 1200/1200 [00:19<00:00, 62.23it/s]


Skipped recordings: 9


In [4]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

X_train: (829483, 1, 625)
y_train: (829483, 2)
X_val: (104736, 1, 625)
y_val: (104736, 2)
X_test: (105176, 1, 625)
y_test: (105176, 2)


In [5]:
print(X_train.dtype)
print(y_train.dtype)


float32
float32


In [6]:
cat ConvTran/Models/model.py

import numpy as np
from torch import nn
from Models.AbsolutePositionalEncoding import tAPE, AbsolutePositionalEncoding, LearnablePositionalEncoding
from Models.Attention import Attention, Attention_Rel_Scl, Attention_Rel_Vec


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class Permute(nn.Module):
    def forward(self, x):
        return x.permute(1, 0, 2)


def model_factory(config):
    if config['Net_Type'][0] == 'T':
        model = Transformer(config, output_size=config['output_size'])
    elif config['Net_Type'][0] == 'CC-T':
        model = CasualConvTran(config, output_size=config['output_size'])
    else:
        model = ConvTran(config, output_size=config['output_size'])
    return model


class Transformer(nn.Module):
    def __init__(self, config, output_size):
        super().__init__()
        # Parameters Initialization -----------------------------------------------
        channel_size, seq_len = config['Data_sh

In [7]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

X: torch.Size([64, 1, 625]) torch.float32
y: torch.Size([64, 2]) torch.float32


In [8]:
config = {
    # Input
    'Data_shape': (32, 1, 625),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [9]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

Input : torch.Size([64, 1, 625])
Output: torch.Size([64, 2])


In [10]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

Prediction shape: torch.Size([64, 2])
Target shape: torch.Size([64, 2])
Loss: 102.15318298339844
Backward pass successful


In [11]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

Optimizer step successful


In [12]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

Loss: 98.30908966064453
Training step successful


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [14]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [15]:
from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [16]:
import os

config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['patience'] = 5
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [ ]:
config['epochs'] = 50

history = train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/50 [00:00<?, ?it/s]2026-10-11 14:58:18,373 | INFO : Validation Summary: epoch: 1.000000 | loss: 24.873045 | SBP_MAE: 42.230713 | DBP_MAE: 8.502048 | SBP_RMSE: 47.759933 | DBP_RMSE: 11.124113 | 
2026-10-11 14:58:18,425 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 64.337781 | SBP_MAE: 94.443336 | DBP_MAE: 35.228645 | SBP_RMSE: 100.229866 | DBP_RMSE: 41.925240 | 
Training Epoch:   2%|▏         | 1/50 [09:50<8:02:15, 590.52s/it]


Best validation loss: 24.873044562711286
Saving best model for epoch: 1



2026-10-11 15:08:08,126 | INFO : Validation Summary: epoch: 2.000000 | loss: 11.115218 | SBP_MAE: 15.450591 | DBP_MAE: 7.756102 | SBP_RMSE: 20.017736 | DBP_RMSE: 10.862632 | 
2026-10-11 15:08:08,190 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 13.027138 | SBP_MAE: 19.289774 | DBP_MAE: 7.743289 | SBP_RMSE: 25.090258 | DBP_RMSE: 10.561213 | 
Training Epoch:   4%|▍         | 2/50 [19:40<7:52:03, 590.07s/it]


Best validation loss: 11.1152179564608
Saving best model for epoch: 2



2026-10-11 15:17:58,166 | INFO : Validation Summary: epoch: 3.000000 | loss: 10.411793 | SBP_MAE: 14.650976 | DBP_MAE: 7.147402 | SBP_RMSE: 18.986864 | DBP_RMSE: 9.896168 | 
2026-10-11 15:17:58,227 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 10.453526 | SBP_MAE: 14.743016 | DBP_MAE: 7.138649 | SBP_RMSE: 19.016897 | DBP_RMSE: 9.958209 | 
Training Epoch:   6%|▌         | 3/50 [29:30<7:42:12, 590.06s/it]

...


Best validation loss: 9.31468597683672
Saving best model for epoch: 18



2026-10-11 21:28:36,241 | INFO : Validation Summary: epoch: 19.000000 | loss: 9.456817 | SBP_MAE: 13.129968 | DBP_MAE: 6.755664 | SBP_RMSE: 17.572498 | DBP_RMSE: 9.321880 | 
2026-10-11 21:28:36,244 | INFO : Epoch 19 Training Summary: epoch: 19.000000 | loss: 8.546602 | SBP_MAE: 11.971372 | DBP_MAE: 6.088455 | SBP_RMSE: 16.298964 | DBP_RMSE: 8.885616 | 
Training Epoch:  38%|███▊      | 19/50 [6:40:08<11:38:38, 1352.22s/it]

2026-10-11 21:38:25,921 | INFO : Validation Summary: epoch: 20.000000 | loss: 9.278189 | SBP_MAE: 12.891356 | DBP_MAE: 6.635665 | SBP_RMSE: 17.337992 | DBP_RMSE: 9.277657 | 
2026-10-11 21:38:26,033 | INFO : Epoch 20 Training Summary: epoch: 20.000000 | loss: 8.506896 | SBP_MAE: 11.916757 | DBP_MAE: 6.063354 | SBP_RMSE: 16.238831 | DBP_RMSE: 8.860767 | 
Training Epoch:  40%|████      | 20/50 [6:49:58<9:21:39, 1123.31s/it] 


Best validation loss: 9.278189440258702
Saving best model for epoch: 20



2026-10-11 21:59:26,964 | INFO : Validation Summary: epoch: 21.000000 | loss: 9.346256 | SBP_MAE: 13.039553 | DBP_MAE: 6.624195 | SBP_RMSE: 17.452202 | DBP_RMSE: 9.234706 | 
2026-10-11 21:59:26,968 | INFO : Epoch 21 Training Summary: epoch: 21.000000 | loss: 8.463715 | SBP_MAE: 11.851626 | DBP_MAE: 6.042224 | SBP_RMSE: 16.165215 | DBP_RMSE: 8.837028 | 
Training Epoch:  42%|████▏     | 21/50 [7:10:59<9:22:53, 1164.62s/it]

In [ ]:
plot_convtran_loss(history)

In [ ]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)

test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [ ]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [ ]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

In [ ]:
# Convert to NumPy arrays if they are PyTorch tensors
if torch.is_tensor(y_true):
    y_true = y_true.detach().cpu().numpy()

if torch.is_tensor(y_pred):
    y_pred = y_pred.detach().cpu().numpy()

# Extract SBP and DBP separately
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

# SBP metrics and plots
print("===== SBP Metrics =====")
print_metrics(sbp_true, sbp_pred)

plot_regression_results(
    sbp_true,
    sbp_pred,
    model_name="ConvTran",
    target_name="SBP"
)

# DBP metrics and plots
print("===== DBP Metrics =====")
print_metrics(dbp_true, dbp_pred)

plot_regression_results(
    dbp_true,
    dbp_pred,
    model_name="ConvTran",
    target_name="DBP"
)

In [ ]:
from processing import evaluate_convtran_vitaldb
convtran_vitaldb_results = evaluate_convtran_vitaldb(
    model=model,  # Your trained, best-checkpoint ConvTran model
    base_dir=BASE_DIR,
    batch_size=64,
    device=device
)